# 01 — Exploration des données C-MAPSS

Objectifs :
1. vérifier la structure des fichiers (26 colonnes) ;
2. décrire la distribution de la durée de vie des moteurs ;
3. visualiser l'évolution des capteurs au fil des cycles ;
4. mesurer les corrélations, avec la RUL et entre capteurs ;
5. repérer les capteurs (quasi) constants **à partir de leur variance**.

Prérequis : `python data/download_data.py`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
SUBSET = "FD001"   # puis "FD004"


In [ ]:
from src.data import load_subset, add_rul, low_variance_sensors, SENSOR_COLS, SETTING_COLS

train, test, rul_test = load_subset(SUBSET)
print("entraînement :", train.shape, " test :", test.shape, " RUL test :", rul_test.shape)
print("moteurs : entraînement", train.unit.nunique(), "· test", test.unit.nunique())
train.head()

In [ ]:
# Aucune valeur manquante attendue : les données sont simulées.
train.isna().sum().sum(), test.isna().sum().sum()

## Durée de vie des moteurs

In [ ]:
life = train.groupby("unit")["cycle"].max()
print(life.describe().round(1))
ax = life.plot.hist(bins=20, edgecolor="white", figsize=(7, 4))
ax.set(xlabel="cycles jusqu'à la panne", ylabel="moteurs", title=f"Durée de vie — {SUBSET}")
plt.tight_layout(); plt.savefig(FIG / f"duree_vie_{SUBSET}.png", dpi=150); plt.show()

## Capteurs constants ou presque

On mesure l'écart-type relatif (écart-type / |moyenne|) de chaque capteur.

In [ ]:
desc = train[SENSOR_COLS].agg(["mean", "std", "min", "max"]).T
desc["std relatif"] = desc["std"] / desc["mean"].abs().replace(0, 1)
display(desc.sort_values("std relatif").round(5))
constants = low_variance_sensors(train)
print("Capteurs retirés (std relatif < 1e-4) :", constants)

## Évolution des capteurs pour quelques moteurs

L'axe horizontal est la RUL (0 = panne), pour aligner les moteurs sur leur fin de vie.

In [ ]:
df = add_rul(train, cap=None)
useful = [c for c in SENSOR_COLS if c not in constants]
units = df.unit.drop_duplicates().sample(5, random_state=0)
fig, axes = plt.subplots(int(np.ceil(len(useful) / 3)), 3, figsize=(14, 2.4 * np.ceil(len(useful) / 3)), sharex=True)
for ax, s in zip(axes.flat, useful):
    for u in units:
        g = df[df.unit == u]
        ax.plot(g.RUL, g[s], lw=0.8)
    ax.set_title(s, fontsize=9); ax.invert_xaxis()
for ax in axes.flat[len(useful):]:
    ax.remove()
fig.supxlabel("RUL (cycles restants)")
plt.tight_layout(); plt.savefig(FIG / f"capteurs_{SUBSET}.png", dpi=150); plt.show()

## Corrélations

In [ ]:
corr_rul = df[useful + ["RUL"]].corr()["RUL"].drop("RUL").sort_values()
corr_rul.plot.barh(figsize=(6, 5), title="Corrélation avec la RUL")
plt.tight_layout(); plt.savefig(FIG / f"correlation_rul_{SUBSET}.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(df[useful].corr(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(useful)), useful, rotation=90); ax.set_yticks(range(len(useful)), useful)
fig.colorbar(im); ax.set_title("Corrélations entre capteurs")
plt.tight_layout(); plt.show()

## Réglages opérationnels

Sur FD002 et FD004, les 3 réglages forment 6 groupes distincts (conditions de vol).

In [ ]:
ax = train.plot.scatter("setting_1", "setting_2", s=2, alpha=0.3, figsize=(6, 4))
ax.set_title(f"Réglages opérationnels — {SUBSET}"); plt.show()

## À retenir pour le rapport

- Nombre de colonnes vérifié : 26.
- Capteurs retirés et justification par la variance : *à compléter*.
- Capteurs les plus corrélés à la RUL : *à compléter*.
- Limite : données simulées, propres, sans valeurs manquantes.